# F1 2026 — Performance & Strategy Analysis

This notebook analyzes the cleaned Australia, China and Japan race dataset.

Questions:
- Which drivers show the strongest race pace?
- Which drivers are most consistent?
- How does tyre age affect lap time?
- Which compounds perform best?
- How do drivers and teams compare across circuits?
- Which drivers gain or lose pace as a race progresses?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("Analysis environment ready.")


## 1. Load the combined dataset


In [ ]:
file_path = PROCESSED_DATA / "f1_2026_three_races.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
display(df.head())


In [ ]:
print("Races:")
print(df["race"].value_counts())

print("\nDrivers:", df["drv"].nunique())
print("Teams:", df["team"].nunique())

print("\nRows by race:")
display(df.groupby("race").size().reset_index(name="lap_records"))


## 2. Prepare valid lap records


In [ ]:
analysis_df = df.copy()

for col in ["time", "lap", "life", "stint", "pos", "s1", "s2", "s3"]:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

analysis_df = analysis_df.dropna(
    subset=["time", "drv", "team", "race"]
).copy()

print("Valid lap records:", len(analysis_df))
display(analysis_df.head())


## 3. Driver race pace


In [ ]:
driver_pace = (
    analysis_df.groupby(["drv", "team"])
    .agg(
        races=("race", "nunique"),
        laps=("time", "count"),
        best_lap=("time", "min"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        pace_std=("time", "std")
    )
    .reset_index()
    .sort_values("average_lap")
)

display(driver_pace)


In [ ]:
plot_df = driver_pace.sort_values("average_lap", ascending=True)

plt.figure(figsize=(11, 7))
plt.barh(plot_df["drv"], plot_df["average_lap"])
plt.xlabel("Average Lap Time (seconds)")
plt.ylabel("Driver")
plt.title("Average Race Pace Across Three 2026 Races")
plt.tight_layout()
plt.show()


## 4. Driver performance by circuit


In [ ]:
race_driver_pace = (
    analysis_df.groupby(["race", "drv", "team"])
    .agg(
        laps=("time", "count"),
        best_lap=("time", "min"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        pace_std=("time", "std")
    )
    .reset_index()
)

display(race_driver_pace.sort_values(["race", "average_lap"]))

pace_matrix = race_driver_pace.pivot_table(
    index="drv", columns="race", values="average_lap"
)
display(pace_matrix)


## 5. Driver consistency


In [ ]:
consistency = (
    analysis_df.groupby(["race", "drv", "team"])
    .agg(
        average_lap=("time", "mean"),
        lap_time_std=("time", "std"),
        laps=("time", "count")
    )
    .reset_index()
)

display(consistency.sort_values(["race", "lap_time_std"]))


## 6. Tyre compound performance


In [ ]:
compound_analysis = (
    analysis_df.dropna(subset=["compound"])
    .groupby(["race", "compound"])
    .agg(
        laps=("time", "count"),
        drivers=("drv", "nunique"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        best_lap=("time", "min")
    )
    .reset_index()
)

display(compound_analysis)

display(
    compound_analysis.pivot_table(
        index="compound",
        columns="race",
        values="average_lap"
    )
)


## 7. Tyre age and degradation


In [ ]:
degradation = (
    analysis_df.dropna(subset=["life"])
    .groupby(["race", "compound", "life"])
    .agg(
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        observations=("time", "count")
    )
    .reset_index()
)

display(degradation.head(40))


In [ ]:
common_compounds = analysis_df["compound"].dropna().value_counts().head(3).index

plt.figure(figsize=(12, 7))

for compound in common_compounds:
    temp = (
        degradation[degradation["compound"] == compound]
        .groupby("life", as_index=False)["average_lap"]
        .mean()
        .sort_values("life")
    )
    plt.plot(temp["life"], temp["average_lap"], marker="o", label=str(compound))

plt.xlabel("Tyre Life")
plt.ylabel("Average Lap Time (seconds)")
plt.title("Average Lap Time vs Tyre Life")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Stint performance


In [ ]:
stint_analysis = (
    analysis_df.dropna(subset=["stint", "compound"])
    .groupby(["race", "drv", "stint", "compound"])
    .agg(
        laps=("lap", "count"),
        average_lap=("time", "mean"),
        best_lap=("time", "min"),
        first_lap_time=("time", "first"),
        last_lap_time=("time", "last"),
        average_tyre_life=("life", "mean")
    )
    .reset_index()
)

stint_analysis["pace_change"] = (
    stint_analysis["last_lap_time"] - stint_analysis["first_lap_time"]
)

display(stint_analysis.head(50))


## 9. Team performance


In [ ]:
team_performance = (
    analysis_df.groupby(["race", "team"])
    .agg(
        drivers=("drv", "nunique"),
        laps=("time", "count"),
        best_lap=("time", "min"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        pace_std=("time", "std")
    )
    .reset_index()
)

display(team_performance.sort_values(["race", "average_lap"]))


## 10. Teammate comparison


In [ ]:
teammate = (
    analysis_df.groupby(["race", "team", "drv"])
    .agg(
        average_lap=("time", "mean"),
        best_lap=("time", "min"),
        laps=("time", "count")
    )
    .reset_index()
)

teammate["team_average"] = (
    teammate.groupby(["race", "team"])["average_lap"].transform("mean")
)

teammate["difference_to_team_average"] = (
    teammate["average_lap"] - teammate["team_average"]
)

display(
    teammate.sort_values(
        ["race", "team", "difference_to_team_average"]
    )
)


## 11. Race progression


In [ ]:
progression = analysis_df.copy()

progression["race_phase"] = pd.cut(
    progression["lap"],
    bins=[0, 15, 30, 1000],
    labels=["Early", "Middle", "Late"],
    include_lowest=True
)

phase_analysis = (
    progression.groupby(["race", "drv", "race_phase"], observed=True)
    .agg(
        average_lap=("time", "mean"),
        laps=("time", "count")
    )
    .reset_index()
)

display(phase_analysis.head(50))


In [ ]:
phase_pivot = phase_analysis.pivot_table(
    index=["race", "drv"],
    columns="race_phase",
    values="average_lap"
)

if {"Early", "Late"}.issubset(phase_pivot.columns):
    phase_pivot["late_minus_early"] = (
        phase_pivot["Late"] - phase_pivot["Early"]
    )

display(phase_pivot.sort_values("late_minus_early" if "late_minus_early" in phase_pivot else phase_pivot.columns[0]))


## 12. Project leaderboard


In [ ]:
leaderboard = (
    analysis_df.groupby(["drv", "team"])
    .agg(
        races=("race", "nunique"),
        laps=("time", "count"),
        average_lap=("time", "mean"),
        best_lap=("time", "min"),
        consistency=("time", "std")
    )
    .reset_index()
)

leaderboard["pace_rank"] = leaderboard["average_lap"].rank(method="min")
leaderboard["consistency_rank"] = leaderboard["consistency"].rank(method="min")

leaderboard = leaderboard.sort_values("pace_rank")
display(leaderboard)


## 13. Export analysis tables


In [ ]:
analysis_output = PROCESSED_DATA / "analysis_outputs"
analysis_output.mkdir(parents=True, exist_ok=True)

exports = {
    "driver_pace.csv": driver_pace,
    "race_driver_pace.csv": race_driver_pace,
    "compound_analysis.csv": compound_analysis,
    "degradation.csv": degradation,
    "stint_analysis.csv": stint_analysis,
    "team_performance.csv": team_performance,
    "teammate_comparison.csv": teammate,
    "race_phase_analysis.csv": phase_analysis,
    "driver_leaderboard.csv": leaderboard,
}

for filename, table in exports.items():
    path = analysis_output / filename
    table.to_csv(path, index=False)
    print("Saved:", path)

print("\nAll analysis tables exported.")


## 14. Final summary


In [ ]:
print("=" * 70)
print("F1 PERFORMANCE & STRATEGY ANALYSIS COMPLETE")
print("=" * 70)
print(f"Races analysed: {analysis_df['race'].nunique()}")
print(f"Drivers analysed: {analysis_df['drv'].nunique()}")
print(f"Teams analysed: {analysis_df['team'].nunique()}")
print(f"Valid lap records: {len(analysis_df):,}")

print("\nTop drivers by average race pace:")
display(
    leaderboard[
        ["drv", "team", "races", "laps", "average_lap", "best_lap", "consistency"]
    ].head(10)
)
